NLP Project

In [ ]:
# Remove existing repo to prevent errors upon session restart
!rm -rf NLP_term_project
!git clone https://github.com/thit2003/NLP_term_project.git

Cloning into 'NLP_term_project'...
remote: Enumerating objects: 28149, done.
remote: Counting objects: 100% (55/55), done.
remote: Compressing objects: 100% (43/43), done.
remote: Total 28149 (delta 17), reused 43 (delta 7), pack-reused 28094 (from 1)
Receiving objects: 100% (28149/28149), 277.85 MiB | 16.39 MiB/s, done.
Resolving deltas: 100% (3003/3003), done.


In [ ]:
import os
import pandas as pd
import numpy as np
import re
from datasets import Dataset, concatenate_datasets, load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding
)

MODEL_NAME = "roberta-base"

LABEL2ID = {"positive": 0, "neutral": 1, "negative": 2}
ID2LABEL = {v: k for k, v in LABEL2ID.items()}

Change file in to Thai

In [ ]:
input_path = "/content/NLP_term_project/wisesight-sentiment/kaggle-competition/test.txt"
output_path = "/content/ThaiTest.txt"


url_pattern = r"http\S+|www\S+"
hashtag_pattern = r"#\S+"
english_pattern = r"[A-Za-z]+"
emoji_pattern = r"[\U00010000-\U0010ffff]"
cleaned_lines = []

with open(input_path, "r", encoding="utf-8") as f:
    lines = f.readlines()

for line in lines:
    content = line.rstrip("\n")
    newline = "\n" if line.endswith("\n") else ""
    content = re.sub(url_pattern, "", content)
    content = re.sub(hashtag_pattern, "", content)
    content = re.sub(emoji_pattern, "", content)
    content = re.sub(english_pattern, "", content)
    content = re.sub(r"\s+", " ", content).strip()
    cleaned_lines.append(content + newline)

final_text = "".join(cleaned_lines)

with open(output_path, "w", encoding="utf-8") as f:
    f.write(final_text)

total_original = len(lines)
total_cleaned = len(cleaned_lines)


with open("/content/ThaiTest.txt", "r", encoding="utf-8") as f:
    text = f.readlines()
print("Text lines (after pandoc):", len(text))
print(f"Original lines: {total_original}, Cleaned lines: {total_cleaned}")


Text lines (after pandoc): 2674
Original lines: 2674, Cleaned lines: 2674


In [ ]:
input_path = "/content/NLP_term_project/wisesight-sentiment/kaggle-competition/train.txt"
output_path = "/content/ThaiTrain.txt"


url_pattern = r"http\S+|www\S+"
hashtag_pattern = r"#\S+"
english_pattern = r"[A-Za-z]+"
emoji_pattern = r"[\U00010000-\U0010ffff]"
cleaned_lines = []

with open(input_path, "r", encoding="utf-8") as f:
    lines = f.readlines()

for line in lines:
    content = line.rstrip("\n")
    newline = "\n" if line.endswith("\n") else ""
    content = re.sub(url_pattern, "", content)
    content = re.sub(hashtag_pattern, "", content)
    content = re.sub(emoji_pattern, "", content)
    content = re.sub(english_pattern, "", content)
    content = re.sub(r"\s+", " ", content).strip()
    cleaned_lines.append(content + newline)

final_text = "".join(cleaned_lines)

with open(output_path, "w", encoding="utf-8") as f:
    f.write(final_text)

total_original = len(lines)
total_cleaned = len(cleaned_lines)


with open("/content/ThaiTrain.txt", "r", encoding="utf-8") as f:
    text = f.readlines()
print("Text lines (after pandoc):", len(text))
print(f"Original lines: {total_original}, Cleaned lines: {total_cleaned}")

Text lines (after pandoc): 24063
Original lines: 24063, Cleaned lines: 24063


In [ ]:
def merge(x,y):
  print("lines: ",len(x))
  print("labels: ",len(y))

  if len(x) != len(y):
    raise ValueError("Text and label files have different number of lines!")

  xlist = []
  ylist = []

  for t,l in zip(x,y):
    t = t.strip()
    l = l.strip()

    if t == "" :
      continue
    if l == "" :
      continue
    ws_map = {"pos": "positive", "neu": "neutral", "neg": "negative", "q": "neutral"}
    l2 = ws_map.get(l)
    xlist.append(t)
    ylist.append(LABEL2ID[l2])

  print("After removing blank text lines:", len(x))
  print("After removing blank text lines:", len(y))

  df = pd.DataFrame({
     "text": xlist,
     "label": ylist
    })
  return df

In [ ]:
with open("/content/ThaiTest.txt", "r", encoding="utf-8") as f:
    text = f.readlines()

with open("/content/NLP_term_project/wisesight-sentiment/kaggle-competition/test_label.txt", "r", encoding="utf-8") as f:
    labels = f.readlines()

with open("/content/ThaiTrain.txt", "r", encoding="utf-8") as f:
    text2 = f.readlines()

with open("/content/NLP_term_project/wisesight-sentiment/kaggle-competition/train_label.txt", "r", encoding="utf-8") as f:
    labels2 = f.readlines()

testdf = merge(text,labels)
traindf = merge(text2,labels2)


output_path1 = "/content/ThaiTest.csv"
output_path2 = "/content/ThaiTrain.csv"
testdf.to_csv(output_path1, index=False, encoding="utf-8-sig")
traindf.to_csv(output_path2, index=False, encoding="utf-8-sig")



lines:  2674
labels:  2674
After removing blank text lines: 2674
After removing blank text lines: 2674
lines:  24063
labels:  24063
After removing blank text lines: 24063
After removing blank text lines: 24063


In [ ]:
train_dataset = Dataset.from_pandas(traindf)
eval_dataset = Dataset.from_pandas(testdf)

In [ ]:
# 1) Tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True, padding=False, max_length=128)

train_dataset = train_dataset.map(tokenize_function, batched=True)
eval_dataset = eval_dataset.map(tokenize_function, batched=True)

train_dataset = train_dataset.rename_column("label", "labels")
eval_dataset = eval_dataset.rename_column("label", "labels")

train_dataset.set_format('torch', columns=['input_ids', 'attention_mask', 'labels'])
eval_dataset.set_format('torch', columns=['input_ids', 'attention_mask', 'labels'])

# 2) Build Model
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=3,
    id2label=ID2LABEL,
    label2id=LABEL2ID,
)

Map:   0%|          | 0/23886 [00:00<?, ? examples/s]

Map:   0%|          | 0/2654 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.weight            | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
classifier.out_proj.weight      | MISSING    | 
classifier.out_proj.bias        | MISSING    | 
classifier.dense.weight         | MISSING    | 
classifier.dense.bias           | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = (preds == labels).mean()

    f1s = []
    for c in [0, 1, 2]:
        tp = np.sum((preds == c) & (labels == c))
        fp = np.sum((preds == c) & (labels != c))
        fn = np.sum((preds != c) & (labels == c))
        precision = tp / (tp + fp + 1e-12)
        recall = tp / (tp + fn + 1e-12)
        f1 = 2 * precision * recall / (precision + recall + 1e-12)
        f1s.append(f1)

    return {"accuracy": float(acc), "f1_macro": float(np.mean(f1s))}

In [ ]:
training_args = TrainingArguments(
    output_dir="./results_Thai",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_steps=100,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    report_to="none",
)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    compute_metrics=compute_metrics,
    processing_class=tokenizer,
)

In [ ]:
print("Starting training...")
trainer.train()

Starting training...


Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 

In [ ]:
metrics = trainer.evaluate()